# Case set up
### Import libraries

In [1]:
import os
import sys
import importlib
import joblib as jl
import numpy as np
import tensorflow as tf

from tqdm import tqdm
from matplotlib import pyplot as plt

### Define PrODE and working case directories paths

In [2]:
prode_dir = os.path.normpath(os.environ['WORKSPACE_PATH'] + '/ML/PrODE/prode/')
working_dir = "./"

In [3]:
if importlib.util.find_spec('prode') is None:
    sys.path.append(prode_dir)
sys.path.append(working_dir)

### Import additional libraries

In [4]:
# PrODE -----------------------------------------------------------------------
from prode.utilities import utils
from prode.apps import functionObjects as fun
from prode.postprocessing import testing, training

# Extra modules ---------------------------------------------------------------
# Inputs
import inputs
from inputs.general import flags
# Physics-related
from physics.data_generator import DataGenerator

### Initialize network

In [5]:
net = fun._net(
    input_module=inputs
)


[PrODE]: Initializing 'IsoThermChemReacNN' network ...


### Initialize PrODE model

In [6]:
for k in ('train', 'save'):
    flags[k] = 0
if flags['load'] not in (1, 2):
    utils.raise_value_err(
        "The network has to be restored from a saved or a checkpointed "
        "one. Modified the 'load' entry in the `flags` input variable."
    )
model = fun._model(
    flags=flags,
    input_module=inputs,
    net=net
)


[PrODE]: Building 'IsoThermChemReacNN' network ...
Model: "IsoThermChemReact_MMOpNN"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 29)]         0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 1)]          0                                            
__________________________________________________________________________________________________
branch_fn (Lambda)              (None, 29)           0           input_1[0][0]                    
__________________________________________________________________________________________________
trunk_fn (Scaling)              (None, 1)            2           input_2[0][0]                    
_______________________

### Generate testing data

In [7]:
# Generate Testing Data ---------------------------------------------------
dataGen_args = utils.get_args_values(
    args=utils.get_args(DataGenerator.__init__),
    modules=[inputs.data, inputs.general]
)
dataGen = DataGenerator(
    **dataGen_args
)
dataGen.data_dir = "./data/"
if not os.path.exists(dataGen.data_dir):
    os.makedirs(dataGen.data_dir)

utils.print_main("Generating testing pts ...")
if inputs.data.from_file:
    test_data = dataGen.testing_data(
        inputs.data.file['test'],
        header=inputs.data.header
    )
else:
    raise NotImplementedError

  0%|          | 0/100 [00:00<?, ?it/s]


[PrODE]: Generating testing pts ...


100%|██████████| 100/100 [02:04<00:00,  1.25s/it]


# Postprocessing - Model analysis

#### Get predictions

In [31]:
utils.print_main("Postprocessing physical quantities ...")
x, y, y_pred = [], [], []
for scenario in tqdm(test_data):
    x_i, y_i = scenario
    x.append(x_i)
    y.append(y_i)
    y_pred.append(model.net.predict(x_i))
n = len(test_data)

  0%|          | 0/100 [00:00<?, ?it/s]


[PrODE]: Postprocessing physical quantities ...


100%|██████████| 100/100 [00:34<00:00,  2.88it/s]


### Set up analysis

In [9]:
path = './model_analysis/'
if not os.path.exists(path):
    os.makedirs(path)
    
FLAGS = {
    'one_pass': 0,
    'speedup': 0,
    'recursive': 1
}

### One-pass inference
Set up

In [10]:
path_ = path + '/onePassInference/'
if not os.path.exists(path_):
    os.makedirs(path_)

ts_label = {}
for i, name in enumerate(inputs.network.names):
    d = 1 if i == 0 else inputs.general.dim['out'][i]
    ts_label[name] = [r'$O$'] \
        + [r'$O_2^{({%s},{%s})}$' % (i+1,j+1) for j in range(d)]

In [11]:
def output_ops(y, timescale):
    if timescale == 0:
        return tf.reverse(y[0], axis=[1])
    else:
        return tf.concat([y[0][:,1:2], y[timescale]], 1)

def single_scenario(path, i):
    path_case = path_+'/case_'+str(i+1).zfill(3)
    for y_scale in ('linear', 'log'):
        path_y = path_case+'/'+y_scale
        if not os.path.exists(path_y):
            os.makedirs(path_y)
        for j, name in enumerate(inputs.network.names):
            x_ = x[i][-1]
            y_ = output_ops(y[i], j)
            y_pred_ = output_ops(y_pred[i], j)
            testing.plot_test(
                path_y,
                x_,
                y_,
                y_pred_,
                ts_label[name],
                fig_name=name,
                labels=[r'$t$ [s]', r'$Y$'],
                scales=['log', y_scale]
            )

if FLAGS['one_pass']:
    utils.print_submain("Testing 'one-pass' inference")
    out = jl.Parallel(n_jobs=-1)(
        jl.delayed(single_scenario)(path_, i) for i in tqdm(range(n))
    )


### Speedup
Set up

In [12]:
path_ = path + '/speedup/'
if not os.path.exists(path_):
    os.makedirs(path_)

t_end = 1.e-7
num_ics = [int(10**i) for i in range(5)]
labels = ['DeepONet', 'Numerical Solver']

In [13]:
path_ = path + '/speedup/'

import time

if FLAGS['speedup']:
    utils.print_submain("Plotting execution time")

    # ODE solver
    dataGen.ODE_sol.t_end = t_end
    test_data = dataGen.testing_data(
        inputs.data.file['test'],
        exec_time=True,
        header=inputs.data.header
    )
    exec_time = []
    for scenario in test_data:
        exec_time.append(scenario[-1])
    ode_time_ref = np.mean(np.array(exec_time))

    # DeepONet & comparison
    exec_time = []
    for n in tqdm(num_ics):
        x_ = [i[np.random.choice(i.shape[0], n, replace=True)] for i in x[-1]]
        start = time.time()
        y_pred = model.net.predict(x_)
#         y_pred = model.net.call(x_)
        end = time.time()
        nn_time = end - start
        ode_time = ode_time_ref * n
        exec_time.append(np.array([nn_time, ode_time]))
    exec_time = np.vstack(exec_time)
    speedup = exec_time[:,1] / exec_time[:,0]

    # Plot style
    style = dict(
        marker='s',
        fillstyle='full',
        markersize=5,
        ls='--',
        lw=1.5
    )

    # Plot execution times
    fig = plt.figure()
    plt.xlabel('\# ODEs solved')
    plt.ylabel('Time [s]')
    plt.loglog(num_ics, exec_time, label=labels,**style)
    plt.legend(fontsize=16)
    plt.show()
    fig.savefig(path_ + 'times.pdf')
    plt.close()

    # Plot speedup
    fig = plt.figure()
    plt.xlabel('\# ODEs solved')
    plt.ylabel('Speedup')
    plt.semilogx(num_ics, speedup, **style)
    fig.savefig(path_ + 'speedup.pdf')
    plt.show()
    plt.close()

### Recursive inference
Set up

In [26]:
path_ = path + '/recursiveInference/'
if not os.path.exists(path_):
    os.makedirs(path_)

ts_label = {}
for i, name in enumerate(inputs.network.names):
    d = 1 if i == 0 else inputs.general.dim['out'][i]
    ts_label[name] = [r'$O$'] \
        + [r'$O_2^{({%s},{%s})}$' % (i+1,j+1) for j in range(d)]

Recursive inference solver

In [27]:
def recursive_inference(
    ic,
    dt0=1.e-7,
    t_end=1.e-3,
    dt_str=1.4,
    dt_max=1.e-4
):
    # Initialize variables
    T = np.array([[0.]])
    Y = tf.concat(
        [1. - np.sum(ic[0:1,2:], keepdims=True), ic[0:1,2:]], axis=1
    )
    # Initialize loop
    t = T
    dt = dt0
    while t <= t_end:
        # Update inputs
        ic_ = np.expand_dims(tf.concat([ic[0,:2], Y[-1,1:]], axis=0), axis=0)
        t_ = np.array([[dt]])
        # Predictions
        y = output_ops(model.net([ic_,t_]), timescale=3)
        # Update time step
        t = t + dt
        dt = min(dt*dt_str, dt_max)
        # Collect data
        T = np.vstack((T, t))
        Y = np.vstack((Y, y))
    return T, Y

Recursive inference plotter

In [28]:
def plot_recursive_inference(
    path,
    x,
    y,
    x_pred,
    y_pred,
    var_names,
    fig_name=None,
    labels=[None, None],
    scales=['log', 'linear']
):
    fig = plt.figure()

    # Set up figure
    x_label, y_label = labels
    x_scale, y_scale = scales

    # X axis
    if x_label is not None:
        plt.xlabel(x_label)
    if x_scale is not None:
        plt.xscale(x_scale)
    plt.xlim([1.e-8, 1.e-3])
#     plt.xlim([min(x_pred), max(x_pred)])

    # Y axis
    if y_label is not None:
        plt.ylabel(y_label)
    if y_scale == 'log':
        plt.yscale(y_scale)
        plt.ylim([np.amin(y)*5.e-1, np.amax(y)*5.e+0])
    else:
        delta = np.amax(y)*0.1
        plt.ylim([np.amin(y)-delta, np.amax(y)+delta])

    if len(y.shape) < 2:
        y = np.expand_dims(y, 1)

    # Plot figure
    # Define styles
    true_style = dict(
        lw=1.
    )
    pred_style = dict(
        marker='x',
        fillstyle='none',
        markersize=5,
        ls='',
        lw=1.
    )

    # Define parameters
    colors = []
    n = y.shape[1]

    # Loop over features
    for d in range(n):

        # >> Plot true values
        plt.plot(
            x,
            y[:,d],
            **true_style
        )
        colors.append(plt.gca().lines[-1].get_color())

        # >> Plot predicted values
        plt.plot(
            x_pred,
            y_pred[:,d],
            **pred_style,
            label=var_names[d],
            c=colors[-1]
        )

    # Plot color legends
    if n > 12:
        fs = 'x-small'
    elif n > 6:
        fs = 'small'
    else:
        fs = 'medium'
    plt.legend(
        [plt.plot([], [], c=colors[i])[0] for i in range(len(var_names))],
        var_names,
        fontsize=fs,
        ncol=int(np.ceil(n/3))
    )

    # Save figure
    fig_path = path + '/' + fig_name + '.pdf'
    fig.savefig(fig_path)
    plt.close()

In [29]:
if FLAGS['recursive']:
    utils.print_submain("Testing 'recursive' inference")
    for i in tqdm(range(n)):
        # Get solution
        t_ = x[i][1]
        y_ = output_ops(y[i], timescale=-1)
        t_pred_, y_pred_ = recursive_inference(x[i][0])
        # Plot solution
        path_case = path_+'/case_'+str(i+1).zfill(3)
        if not os.path.exists(path_case):
            os.makedirs(path_case)
        for y_scale in ('linear', 'log'):
            plot_recursive_inference(
                path_case,
                t_,
                y_,
                t_pred_,
                y_pred_,
                list(ts_label.values())[-1],
                fig_name=y_scale,
                labels=[r'$t$ [s]', r'$Y$'],
                scales=['log', y_scale]
            )

  0%|          | 0/100 [00:00<?, ?it/s]

[PrODE]:   Testing 'recursive' inference


100%|██████████| 100/100 [08:38<00:00,  5.19s/it]
